# 25 (MLA) — Regression Workflow

**ML Analyst perspective.** Train a `LinearRegression` on `risco_score`, predict on the held-out test set via SQL pushdown, and evaluate with MAE/MSE/RMSE/R². The model is fit in numpy; scoring never moves data to Python.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Load the prepared splits

Read the tables persisted in notebook 24.

In [ ]:
train = session.table("mla_train")
test = session.table("mla_test")
print("train:", train.count(), "| test:", test.count())

## 2. Train the model

`LinearRegression` fits in numpy (closed-form normal equations, ridge-regularized).

In [ ]:
from irispark.ml.regression import LinearRegression

feats = ["renda_std", "idade_std", "historico_idx_std", "divida_ratio_std"]
lr = LinearRegression(featuresCol=feats, labelCol="risco_score", regParam=0.1)
model = lr.fit(train)
print("backend:", model.backend)
print("intercept:", round(model.intercept, 3))
for c, w in zip(feats, model.coefficients):
    print(f"  {c}: {w:.3f}")

## 3. Predict on the test set

`transform` pushes the coefficients back as a SQL expression — no data leaves IRIS.

In [ ]:
pred = model.transform(test)
pred.select("cliente_id", "risco_score", "prediction").show(5)

## 4. Evaluate

MAE, MSE, RMSE, R² — all computed in SQL.

In [ ]:
from irispark.ml.evaluation import RegressionEvaluator

for m in ("mae", "mse", "rmse", "r2"):
    ev = RegressionEvaluator(predictionCol="prediction", labelCol="risco_score", metricName=m)
    print(f"{m}: {ev.evaluate(pred):.3f}")

## 5. Baseline comparison

A model that always predicts the mean is the floor. The R² already tells us how much better we are.

In [ ]:
from irispark.functions import avg

mean_risco = train.select(avg("risco_score").alias("m")).first()["m"]
print("mean predictor R² would be 0; our model beats it if R² > 0")

## 6. What's next

As the estimator portfolio grows (trees, KMeans — ml_scope Phase 8/10), the same fit → predict → evaluate loop applies unchanged.

In [ ]:
print("regression workflow complete")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")